### Homework 2 Alex Khvatov

In [ ]:
from embedder import Embedder
embed = Embedder(path="models/Xenova/all-MiniLM-L6-v2")



#### Q1. Embedding a query

In [6]:
query = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query)

In [ ]:
v_query[0]
#-0.02

np.float64(-0.020582036807885073)

#### Loading the data

In [10]:
from gitsource import GithubRepositoryDataReader


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

#### Q2. Cosine similarity

In [12]:
doc = next(
    (d for d in documents if '02-vector-search/lessons/07-sqlitesearch-vector.md' in d['filename'])
)

In [15]:
v_content = embed.encode(doc['content'])


In [ ]:
#cosine similarity
import numpy as np
v1 = np.array(v_query)
v2 = np.array(v_content)
cosine_similarity = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
cosine_similarity

#0.37

np.float64(0.3610702803026061)

#### Q3. Chunking and search by hand

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

In [18]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [39]:
vectors = []
for i in chunks:
    content = i['content']
    batch_vectors = embed.encode(content)
    vectors.extend([batch_vectors])

In [40]:
np_vectors = [np.array(v) for v in vectors]

In [41]:
np_vectors
X = np.array(np_vectors)

In [45]:
scores = X.dot(v1)
scores

array([ 3.15187667e-01,  2.01479590e-01,  5.90558833e-02, -7.67662092e-02,
        1.18452457e-01, -1.41697008e-01, -2.81406239e-02, -4.65670219e-02,
       -2.06994421e-02, -6.07744072e-02,  2.13273697e-01,  8.87601283e-02,
       -1.97269148e-02,  3.11629945e-01,  5.51034649e-01,  2.04008064e-01,
        2.12515834e-01,  1.93649166e-01,  2.51961211e-01,  1.31078657e-01,
        2.59120526e-01,  7.63816029e-02,  9.59192920e-02,  9.81469839e-03,
       -3.59106955e-02,  1.01211556e-02,  1.10786957e-01, -9.90259327e-02,
       -3.71170699e-02,  7.59057318e-02, -3.35340264e-02,  8.86838363e-03,
        1.02636359e-01,  6.89614769e-02,  1.29408804e-01,  2.57709174e-01,
        3.23680535e-01,  1.06350088e-01,  5.61891403e-02,  2.34017413e-01,
        1.97954389e-01,  9.64296226e-02,  1.93709947e-01,  2.16719278e-01,
        3.48340450e-01,  5.10906387e-02,  2.05212841e-01,  1.05416126e-01,
       -3.25432324e-02,  4.94665571e-02,  2.38574833e-01, -3.44206606e-02,
        1.82165502e-01,  

In [46]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.648901732433228))

In [48]:
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

#### Q4. Vector search with minsearch

In [49]:
query='What metric do we use to evaluate a search engine?'